In [1]:
# Setup
import duckdb
import numpy as np
import pandas as pd

LST = "../data/silver/nyc/listings.parquet"
CAL = "../data/silver/nyc/calendar.parquet"

df = duckdb.sql(f"SELECT * FROM read_parquet('{LST}')").df()
priced = df[df["price"].notna()].copy()
priced["stay_type"] = np.where(priced["quote_nights"] < 28,
                               "short stay", "monthly stay")
priced["log10_price"] = np.log10(priced["price"])

In [2]:
# What do listing names look like when bedrooms are missing?
homes = priced[priced["room_type"] == "Entire home/apt"]
missing = homes[homes["bedrooms"].isna()]

print("Entire homes missing bedrooms:", len(missing))
print()
for name in missing["name"].head(10):
    print(name)

names = missing["name"].fillna("").str.lower()
print()
print("Name mentions 'studio':       ",
      round(names.str.contains("studio").mean() * 100, 1), "%")
print("Name mentions 'N bedroom(s)': ",
      round(names.str.contains(r"\d+\s*bedroom").mean() * 100, 1), "%")

Entire homes missing bedrooms: 1743

Sunny Williamsburg Loft with Private Cedar Sauna
Midtown Pied-a-terre
Beautiful Harlem Studio Apartment - private access
Truly Amazing Oasis In Williamsburg
Charming furnished Studio-Loft
Chelsea studio with balcony & views
Private Studio Near NY-Presbyterian Hospital
Solo Travelers Private Garden XL Studio
Stylish Designer Studio with Piano
Spacious, Furnished Unit with Private Backyard

Name mentions 'studio':        67.8 %
Name mentions 'N bedroom(s)':  1.4 %


In [3]:
# A fair outlier rule
# Quartiles of log price within each room type + stay type segment
group = priced.groupby(["room_type", "stay_type"])["log10_price"]
q1 = group.transform(lambda s: s.quantile(0.25))
q3 = group.transform(lambda s: s.quantile(0.75))
iqr = q3 - q1

# Flag prices far outside the normal range of their own segment
K = 3
priced["price_outlier"] = (
    (priced["log10_price"] > q3 + K * iqr)
    | (priced["log10_price"] < q1 - K * iqr)
)

print("Outliers flagged:", priced["price_outlier"].sum())
print()
print(priced.groupby(["room_type", "stay_type"])["price_outlier"].sum())
print()
print(
    priced[priced["price_outlier"]]
    .sort_values("price")[["room_type", "stay_type", "accommodates", "price"]]
    .to_string()
)

Outliers flagged: 103

room_type        stay_type   
Entire home/apt  monthly stay    37
                 short stay       2
Hotel room       monthly stay     0
                 short stay       0
Private room     monthly stay    54
                 short stay       8
Shared room      monthly stay     2
                 short stay       0
Name: price_outlier, dtype: int64

             room_type     stay_type  accommodates     price
15934     Private room  monthly stay             2      4.58
18314     Private room  monthly stay             1      7.20
1688      Private room  monthly stay             2      7.22
23558  Entire home/apt  monthly stay             4     10.45
22893  Entire home/apt  monthly stay             2     10.75
18464  Entire home/apt  monthly stay             2     11.07
18736  Entire home/apt  monthly stay             2     12.33
20092     Private room  monthly stay             4    669.97
18997     Private room  monthly stay             2    680.52
17729     Priv

In [4]:
# Remove the booking-window effect
open_calendar = duckdb.sql(f"""
    WITH active AS (
        SELECT listing_id,
               CASE WHEN quote_nights < 28 THEN 'short stay'
                    ELSE 'monthly stay' END AS stay_type
        FROM read_parquet('{LST}')
        WHERE price IS NOT NULL
    ),
    -- listings with at least one open night at days 330-364
    -- (their booking window reaches about 12 months)
    open_listings AS (
        SELECT listing_id
        FROM read_parquet('{CAL}')
        WHERE days_from_start >= 330
        GROUP BY listing_id
        HAVING BOOL_OR(is_available)
    )
    SELECT
        FLOOR(c.days_from_start / 30) * 30 AS days_ahead_from,
        a.stay_type,
        COUNT(DISTINCT c.listing_id)       AS listings,
        ROUND(AVG(CASE WHEN c.is_available = FALSE THEN 100.0
                       WHEN c.is_available = TRUE  THEN 0.0 END), 1)
                                           AS pct_unavailable
    FROM read_parquet('{CAL}') c
    JOIN active a        ON c.listing_id = a.listing_id
    JOIN open_listings o ON c.listing_id = o.listing_id
    WHERE c.days_from_start < 330
    GROUP BY 1, 2
    ORDER BY 1, 2
""").df()

print(open_calendar.pivot(index="days_ahead_from", columns="stay_type",
                          values=["listings", "pct_unavailable"]))

                    listings            pct_unavailable           
stay_type       monthly stay short stay    monthly stay short stay
days_ahead_from                                                   
0.0                  10776.0     2652.0            60.8       52.9
30.0                 10776.0     2652.0            44.1       36.3
60.0                 10776.0     2652.0            26.3       27.3
90.0                 10776.0     2652.0            21.7       26.5
120.0                10776.0     2652.0            17.5       20.6
150.0                10776.0     2652.0            13.7       17.0
180.0                10776.0     2652.0             9.2       16.4
210.0                10776.0     2652.0             4.7        5.5
240.0                10776.0     2652.0             3.4        5.0
270.0                10776.0     2652.0             2.5        5.1
300.0                10776.0     2652.0             1.5        4.2
